# First-Level fMRI GLM Analysis

This notebook runs the flexibility-stability **first-level GLM** using `fmri_first_level.py`.

## Requirements

Install dependencies (once):

```bash
pip install -r requirements.txt
```

## Expected inputs

| Input | Description |
|-------|-------------|
| BIDS root | Contains `sub-*/func/*_events.tsv` |
| Derivatives root | Contains fMRIPrep `sub-*/func/*_desc-preproc_bold.nii.gz` and confounds |
| Output directory | Will be created if missing |

## Outputs (per subject)

- `model/` — design matrix TSV
- `figures/` — design matrix and correlation plots
- `betas/` — condition beta and variance maps (`*_beta.nii.gz`, `*_varbeta.nii.gz`)

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

    from fmri_first_level import (
    FirstLevelConfig,
    configure_logging,
    discover_subjects,
    run_batch,
    run_subject,
    )

    configure_logging(verbose=False)
    print(f"Notebook directory: {NOTEBOOK_DIR}")

## Configuration

Edit the paths below to match your local BIDS layout.

- Set `TEST_SINGLE_SUBJECT = True` to run one subject first.
- Set `N_JOBS = 1` for debugging; increase for parallel batch processing.

In [ ]:
# --- User configuration ---

# Project root is three levels above this notebook directory by default:
# full_factorial -> first_level -> derivatives -> SF
PROJECT_ROOT = NOTEBOOK_DIR.parents[3]

BIDS_ROOT = PROJECT_ROOT # change if raw BIDS lives elsewhere
DERIVATIVES_DIR = PROJECT_ROOT / "derivatives"
OUTPUT_DIR = NOTEBOOK_DIR # save outputs next to this notebook

TASK = "flexibilitystability"
SPACE = "MNI152NLin2009cAsym"
RUNS = [1]
N_JOBS = 1

TEST_SINGLE_SUBJECT = True
TEST_SUBJECT = "sub-110" # used only when TEST_SINGLE_SUBJECT is True
SUBJECTS = None # e.g. ["sub-110", "sub-111"]; None = all subjects under BIDS_ROOT

config = FirstLevelConfig(
bids_root=BIDS_ROOT,
derivatives_dir=DERIVATIVES_DIR,
output_dir=OUTPUT_DIR,
task=TASK,
space=SPACE,
runs=RUNS,
subjects=[TEST_SUBJECT] if TEST_SINGLE_SUBJECT else SUBJECTS,
n_jobs=N_JOBS,
)

print("BIDS root:", config.bids_root)
print("Derivatives dir:", config.derivatives_dir)
print("Output dir:", config.output_dir)

## Validate inputs

In [ ]:
subjects = discover_subjects(config.bids_root, config.subjects)
if not subjects:
    raise FileNotFoundError(
f"No subjects found under {config.bids_root}. "
"Update BIDS_ROOT in the configuration cell."
)

missing = []
for subject in subjects:
    for run in config.runs:
        event_path = config.bids_root / subject / "func" / (
        f"{subject}_task-{config.task}_run-{run}_events.tsv"
        )
        bold_path = (
        config.derivatives_dir / subject / "func" /
        f"{subject}_task-{config.task}_run-{run}_space-{config.space}_desc-preproc_bold.nii.gz"
        )
        confounds_path = (
        config.derivatives_dir / subject / "func" /
        f"{subject}_task-{config.task}_run-{run}_desc-confounds_timeseries.tsv"
        )
        for path in (event_path, bold_path, confounds_path):
            if not path.exists():
                missing.append(str(path))

                print(f"Subjects to process ({len(subjects)}):", subjects)
            if missing:
                print("\nMissing files (first 10):")
            for path in missing[:10]:
                print(" -", path)
                raise FileNotFoundError(
                f"{len(missing)} required file(s) missing. Update BIDS_ROOT or DERIVATIVES_DIR."
                )

                print("All required inputs found.")

## Run first-level GLM

In [ ]:
config.output_dir.mkdir(parents=True, exist_ok=True)

if TEST_SINGLE_SUBJECT:
    completed = [run_subject(config, TEST_SUBJECT)]
else:
    completed = run_batch(config)

    print("Completed subjects:", completed)

## Inspect outputs (optional)

In [ ]:
import pandas as pd

inspect_subject = TEST_SUBJECT if TEST_SINGLE_SUBJECT else subjects[0]
subject_dir = config.output_dir / inspect_subject

design_files = sorted((subject_dir / "model").glob("*design_matrix.tsv"))
beta_files = sorted((subject_dir / "betas").glob("*_beta.nii.gz"))
figure_files = sorted((subject_dir / "figures").glob("*.png"))

print(f"Subject: {inspect_subject}")
print(f"Design matrices: {len(design_files)}")
print(f"Beta maps: {len(beta_files)}")
print(f"Figures: {len(figure_files)}")

if design_files:
    dm = pd.read_csv(design_files[0], sep="\t")
    print("\nDesign matrix shape:", dm.shape)
    print("Condition regressors:")
for col in dm.columns[:12]:
    print(" -", col)